In [ ]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
import torch
import matplotlib.pylab as plt
from synthetic_observations import Observations
from gaussian_synthetic_observations import Gaussian_Observations
from transformer import *
from spectrum_lsf import Score_Likelihood
from score_models import ScoreModel
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from template import Template
from sbart_rv_finder import RV_Retrieval
from mala import MALA

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
change_res=True

In [ ]:
# Load model
checkpoints_dir = 'gausb64nf64ch2_spir4096moredownsample'
model = ScoreModel(checkpoints_directory=checkpoints_dir,device=device)

# This notebook is to get the joint sampling to work

## First let's create the observations

In [ ]:
snr = 200
N = 10
obs = Observations(i=0,SNR=snr,filepath='../data/spirou_4096_val.df',N=N)
synthetic_spectra, uncertainty = obs.make_observations(func='connors',add_RV=True,change_res=change_res)
synthetic_spectra, uncertainty = obs.post_process()

## Now let's get the template and it's RVs

In [ ]:
# Create template
temp = Template(synthetic_spectra[:,:],obs.berv[:],obs.inst_wgrid,obs.wgrid)
template = temp.make_template(func='scipy')

In [ ]:
# Find template RVs
sbart = RV_Retrieval(snr,template,obs.wgrid,obs.inst_wgrid,N)
templatervs, uncs = sbart.find_dv(synthetic_spectra[:,:].cpu(),uncertainty.cpu()[:,:],obs.berv[:].cpu(),func='connors')
templatervs

## Now let's sample for a spectrum that is intialized with template rvs

In [ ]:
# Initialize with the following 
B = 2
bervs_to_send = obs.berv.unsqueeze(0).expand(B,N)
planetrv_to_send = torch.tensor(templatervs).to(DEVICE).unsqueeze(0).expand(B,N)

In [ ]:
from torch.autograd.functional import jacobian

def f_wrapped(x):
    return forward_model(x, obs.wgrid, obs.inst_wgrid, obs.berv.unsqueeze(0), obs.planet.unsqueeze(0), change_res)
x = obs.training
# Compute jacobian of shape  [B,N,L,B,1,D]
A_full = jacobian(f_wrapped, x, create_graph=True)
A = A_full[0,:,:,0,0,:]

In [ ]:
%reload_ext autoreload

In [ ]:
# Prior Sampling 

LSF = Score_Likelihood(Y=synthetic_spectra,V= planetrv_to_send,sig_n=uncertainty, spec_wgrid=obs.wgrid, inst_wgrid=obs.inst_wgrid, SNR=snr, berv=bervs_to_send,beta_min=1e-2,
                    beta_max=20,A=A,change_res=change_res)
dimensions = [1,len(obs.wgrid)]
posterior_samples = model.sample(shape=[B, *dimensions], steps=3000,likelihood_score_fn=LSF)
posterior_samples_np = posterior_samples.cpu()

# MALA testing

In [ ]:
plt.plot(posterior_samples[1,0].cpu())

In [ ]:
mala = MALA(synthetic_spectra[:,:],uncertainty[:,:],obs.berv,snr,obs.inst_wgrid,obs.wgrid)

torchtemplatervs = torch.tensor(templatervs,dtype=torch.float64).to(DEVICE).unsqueeze(0).expand(B,N)

samples, accepted = mala.find_rv(torchtemplatervs,posterior_samples,1000)

In [ ]:
mala = MALA(synthetic_spectra[:,:],uncertainty[:,:],obs.berv,snr,obs.inst_wgrid,obs.wgrid)

torchtemplatervs = torch.tensor(templatervs,dtype=torch.float64).to(DEVICE).unsqueeze(0).expand(B,N)

samples_int, accepted_int = mala.find_rv(torchtemplatervs,obs.training.repeat(B,1,1),1000)

In [ ]:
torch.sum(accepted,dim=0)/1000

In [ ]:
plt.plot(samples[:,0,0].cpu())

In [ ]:
torch.sqrt(torch.mean((torch.tensor(templatervs).cpu()-obs.planet[:].cpu())**2))

In [ ]:
rvs = torch.mean(samples[50:,:,:],dim=0)[0]
torch.sqrt(torch.mean((rvs-obs.planet[:])**2))

In [ ]:
rvs_int = torch.mean(samples_int[50:,:,:],dim=0)[0]
torch.sqrt(torch.mean((rvs_int-obs.planet[:])**2))

In [ ]:
fix, axs = plt.subplots(3,3,figsize=(10,10))
axs = axs.flatten()  # Flatten to easily index from 0 to 8

for i in range(9):
    n, _,_ = axs[i].hist(samples[50:,1,i].cpu(),density=True,alpha=0.6)
    axs[i].vlines(obs.planet[i].cpu(),0,max(n)+0.02,color="k",lw=2)
    axs[i].vlines(templatervs[i],0,max(n)+0.02,color="orange",lw=2)

In [ ]:
samples.shape